# Notebook 1 — Demo walkthrough: Agent & Vector store

This notebook re-runs the lab demo (`lab-agent-vector-store.ipynb`) step by step, with a short plain-language note before each step.
The original demo file is **not changed**.

**The idea in one sentence:** we put two different knowledge sources into two vector stores, turn each into a question-answering *tool*,
and give both tools to an *agent* that reads the tool descriptions and decides which one to ask.

### Small changes from the demo (so it runs locally on Python 3.14)
| Demo | Here | Why |
|---|---|---|
| `from langchain.chains ...`, `from langchain.agents ...` | `from langchain_classic...` | LangChain 1.x moved these older classes to the `langchain-classic` package |
| `Chroma` vector store | `InMemoryVectorStore` | Same job (store vectors, find similar ones), already built into LangChain, no extra install |
| `OpenAI()` → `gpt-3.5-turbo-instruct` | `ChatOpenAI(model="gpt-4o-mini")` | OpenAI retired `gpt-3.5-turbo-instruct` (the API now returns 404) |
| `func=state_of_union.run` | `func=ask_state_of_union` (a tiny wrapper function) | A Python 3.14 bug in LangChain crashes when it inspects `chain.run` |
| `https://beta.ruff.rs/docs/faq/` | `https://docs.astral.sh/ruff/faq/` | The old address now just redirects there |


## Step 0 — Speed fix for this PC (not part of the lab)

On this PC Python tries the IPv6 route to OpenAI first, waits ~10 s for it to fail, then uses IPv4 — so every call took ~40 s.
These 3 lines tell Python to try IPv4 addresses first. It only affects this notebook while it runs; nothing on the system changes.

In [1]:
# Fix for this PC: try IPv4 before IPv6, otherwise every OpenAI call waits ~40 s
# (safe to run more than once: the real function is saved only the first time)
import socket
if not hasattr(socket, "_original_getaddrinfo"):
    socket._original_getaddrinfo = socket.getaddrinfo
socket.getaddrinfo = lambda *args, **kwargs: sorted(
    socket._original_getaddrinfo(*args, **kwargs), key=lambda addr: addr[0] != socket.AF_INET
)

## Step 1 — Imports

- `RetrievalQA`: a ready-made chain = *search the vector store* → *paste the found chunks into a prompt* → *ask the LLM*.
- `InMemoryVectorStore`: keeps the chunk vectors in RAM and finds the ones closest to a question.
- `ChatOpenAI` is the LLM, `OpenAIEmbeddings` turns text into vectors.
- `CharacterTextSplitter` cuts long text into chunks; `TextLoader` / `WebBaseLoader` read a file / a web page.

In [2]:
import os
os.environ.setdefault("USER_AGENT", "ironhack-lab")  # WebBaseLoader warns if no user agent is set

from langchain_classic.chains import RetrievalQA
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader, WebBaseLoader

C:\Users\Windows\AppData\Local\Temp\ipykernel_10056\1376232375.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, WebBaseLoader


## Step 2 — API key and the LLM

The key is read from the `OPENAI_API_KEY` environment variable (already set on this PC; `load_dotenv` would also pick up a `.env` file if there was one).
`temperature=0` makes the answers as repeatable as possible.

In [3]:
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("key found:", OPENAI_API_KEY is not None)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)

key found: True


## Step 3 — Knowledge source 1: the State of the Union speech

1. **Load** the text file into a `Document`.
2. **Split** it into chunks of ~1000 characters (the splitter cuts at blank lines, so a few chunks come out a bit larger — that is what the *"Created a chunk of size …"* messages mean; harmless).
3. **Embed** every chunk (one OpenAI call turns each chunk into a list of numbers) and **store** the vectors.

In [4]:
doc_path = "./datasets/state_of_the_union.txt"

loader = TextLoader(doc_path, encoding="utf-8")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
print(len(texts), "chunks")

embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)
docsearch = InMemoryVectorStore.from_documents(texts, embeddings)

42 chunks


Peek inside: ask the vector store directly (no LLM yet) which chunk is closest to a question.

In [5]:
hit = docsearch.similarity_search("Ketanji Brown Jackson", k=1)[0]
print(hit.page_content[:500])

Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. 

Tonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. 

One of the most serious constitutional respon


Now wrap the store in a `RetrievalQA` chain. `chain_type="stuff"` means: *stuff* the top chunks (4 by default) straight into the prompt.

In [6]:
state_of_union = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)
state_of_union.invoke({"query": "What did the president say about Ketanji Brown Jackson?"})["result"]

"The president described Ketanji Brown Jackson as one of the nation's top legal minds and stated that she will continue Justice Breyer's legacy of excellence. He highlighted her background as a former top litigator in private practice, a former federal public defender, and noted that she comes from a family of public school educators and police officers. He also mentioned that she is a consensus builder and has received a broad range of support from various organizations, including the Fraternal Order of Police and former judges appointed by both Democrats and Republicans."

## Step 4 — Knowledge source 2: the Ruff FAQ web page

Same recipe, but the text comes from a web page. Ruff is a fast Python linter.

In [7]:
loader = WebBaseLoader("https://docs.astral.sh/ruff/faq/")
docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
print(len(ruff_texts), "chunks")

ruff_db = InMemoryVectorStore.from_documents(ruff_texts, embeddings)
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

Created a chunk of size 2209, which is longer than the specified 1000
Created a chunk of size 3190, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 2313, which is longer than the specified 1000


23 chunks


## Step 5 — Turn each chain into a tool

An agent can only call *tools*. A `Tool` has:
- a **name**,
- a **func** that takes a string question and returns a string answer,
- a **description** — the agent reads only this to decide which tool fits. Good descriptions = good routing.

The two small `ask_...` functions are our workaround for the Python 3.14 bug (they do exactly what `chain.run` did).

In [8]:
from langchain_classic.agents import AgentType, Tool, initialize_agent

def ask_state_of_union(question: str) -> str:
    return state_of_union.invoke({"query": question})["result"]

def ask_ruff(question: str) -> str:
    return ruff.invoke({"query": question})["result"]

In [9]:
tools = [
    Tool(
        name="State of Union QA System",
        func=ask_state_of_union,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ask_ruff,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

## Step 6 — Agent as *reasoner*

`ZERO_SHOT_REACT_DESCRIPTION` = a **ReAct** agent: it loops *Thought → Action (pick a tool + input) → Observation (tool answer)* until it can write a *Final Answer*.
"Zero-shot" means it chooses tools from their descriptions alone, without examples.
`verbose=True` prints that loop — read the green/yellow log to see which tool it picked.
(LangChain prints a deprecation warning here: `initialize_agent` is the older API. It still works and matches the lab.)

In [10]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

C:\Users\Windows\AppData\Local\Temp\ipykernel_10056\1099849740.py:1: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent = initialize_agent(


In [11]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)



> Entering new AgentExecutor chain...
I need to find out what President Biden said about Ketanji Brown Jackson in the most recent State of the Union address. 
Action: State of Union QA System
Action Input: What did President Biden say about Ketanji Brown Jackson in the State of the Union address?
Observation: President Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson, describing her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence.
Thought:I now know the final answer.  
Final Answer: President Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson, describing her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence.

> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': "President Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson, describing her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence."}

In [12]:
agent.invoke("Why use ruff over flake8?")



> Entering new AgentExecutor chain...
I need to compare the features and benefits of Ruff and Flake8 to provide a comprehensive answer about why one might choose Ruff over Flake8. 
Action: Ruff QA System
Action Input: Why should someone use Ruff over Flake8?
Observation: Someone might choose to use Ruff over Flake8 for several reasons:

1. **Performance**: Ruff is designed to be faster than Flake8, providing quicker feedback on lint violations.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, including many popular Flake8 plugins natively, which means it can cover a broader range of linting issues without needing additional plugins.

3. **Automatic Fixing**: Ruff is capable of automatically fixing its own lint violations, which Flake8 does not support.

4. **No Custom Rules**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins in Rust, which can simplify the setup and reduce conflicts.

5. **Compatibility with Formatters**: Ruff is des

{'input': 'Why use ruff over flake8?',
 'output': 'Someone might choose to use Ruff over Flake8 for its performance, comprehensive rule set, automatic fixing capabilities, compatibility with formatters, and its potential as a drop-in replacement for Flake8.'}

**What to notice:** after the tool's *Observation*, the agent takes one more *Thought* and writes its own *Final Answer* — an extra LLM call.

## Step 7 — Agent as *router* (`return_direct=True`)

With `return_direct=True` the agent stops right after the tool answers and returns that answer unchanged — it only picks the route.

In [13]:
tools = [
    Tool(
        name="State of Union QA System",
        func=ask_state_of_union,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ask_ruff,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [14]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)



> Entering new AgentExecutor chain...
I need to find out what President Biden said about Ketanji Brown Jackson in the most recent State of the Union address. 
Action: State of Union QA System
Action Input: What did Biden say about Ketanji Brown Jackson in the State of the Union address?
Observation: In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the speech. He described her as one of the nation's top legal minds who will continue Justice Stephen Breyer's legacy of excellence.


> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': "In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the speech. He described her as one of the nation's top legal minds who will continue Justice Stephen Breyer's legacy of excellence."}

In [15]:
agent.invoke("Why use ruff over flake8?")



> Entering new AgentExecutor chain...
I need to gather information about the advantages of using Ruff compared to Flake8, which are both Python linters. This will help me understand the reasons one might prefer Ruff over Flake8. 
Action: Ruff QA System
Action Input: "Why use ruff over flake8?"
Observation: Ruff can be preferred over Flake8 for several reasons:

1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, which is written in Python.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, including many from popular Flake8 plugins, while Flake8 has around 409 total rules. This means Ruff can catch a wider variety of issues.

3. **Automatic Fixing**: Ruff is capable of automatically fixing its own lint violations, which Flake8 does not support.

4. **No Custom Rules**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, which can simplify the setup.

5. **Compatibility with Forma

{'input': 'Why use ruff over flake8?',
 'output': 'Ruff can be preferred over Flake8 for several reasons:\n\n1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, which is written in Python.\n\n2. **Comprehensive Rule Set**: Ruff implements over 900 rules, including many from popular Flake8 plugins, while Flake8 has around 409 total rules. This means Ruff can catch a wider variety of issues.\n\n3. **Automatic Fixing**: Ruff is capable of automatically fixing its own lint violations, which Flake8 does not support.\n\n4. **No Custom Rules**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, which can simplify the setup.\n\n5. **Compatibility with Formatters**: Ruff is designed to work well with formatters like Black, and it can be used alongside them without conflicts.\n\n6. **Drop-in Replacement**: Ruff can serve as a drop-in replacement for Flake8 under certain conditions, making it easier for 

**What to notice:** the log ends right after the *Observation* — no final *Thought*. Faster and cheaper, but the agent can't combine two sources.

## Step 8 — Multi-hop question (needs both tools)

The question has two parts: first find something in the Ruff docs, then check it in the speech.
So we go back to `return_direct=False`. The descriptions also say *"not referencing any obscure pronouns"* so the agent writes self-contained questions for each tool.

In [16]:
tools = [
    Tool(
        name="State of Union QA System",
        func=ask_state_of_union,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ask_ruff,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [17]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"
)



> Entering new AgentExecutor chain...
I need to find out what tool Ruff uses to run over Jupyter Notebooks and whether the president mentioned that tool in the State of the Union address. I'll start by checking the Ruff QA System for information about the tool used by Ruff.

Action: Ruff QA System  
Action Input: What tool does Ruff use to run over Jupyter Notebooks?  
Observation: Ruff integrates with nbQA, a tool for running linters and code formatters over Jupyter Notebooks.
Thought:Now that I know Ruff uses nbQA to run over Jupyter Notebooks, I need to check if the president mentioned nbQA in the State of the Union address.

Action: State of Union QA System  
Action Input: Did the president mention nbQA in the State of the Union address?  
Observation: I don't know.
Thought:I need to find out if the president mentioned nbQA in the State of the Union address, but the previous action did not provide an answer. I will try rephrasing the question to see if I can get a clearer respons

{'input': 'What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?',
 'output': 'Ruff uses nbQA to run over Jupyter Notebooks, but the president did not mention nbQA in the State of the Union address.'}

**What to notice:** two *Action* steps — first Ruff, then State of the Union with the tool name filled in — and one combined *Final Answer*.

## Summary of the workflow (we reuse it in notebook 2)
1. Load text → 2. split into chunks → 3. embed + store in a vector store → 4. `RetrievalQA` chain per store →
5. wrap each chain in a `Tool` with a clear description → 6. `initialize_agent` → 7. ask direct, router and multi-hop questions.